# Predicting FLS2-flg22 Perception
The purpose of this notebook is to test several ML classification algorithms on the Li dataset (Perception vs. No Perception). This serves as a proof of concept for testing feature ranking and several different algorithms using a dataset we know will be simple to classify. Next, we will use similar methods to classify Immunogenic vs. Antagonist flg22 peptides.  

---
### Importing Libraries and Data

In [1]:
import numpy as np
import pandas as pd

from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

In [2]:
df = pd.read_csv("alphafold_scores_with_known.csv")

reps = [1,2,3]
features = [
    "ptm",
    "iptm",
    "FLS2-flg22 iptm",
    "FLS2-flg22 pae",
    "BAK1-flg22 iptm",
    "BAK1-flg22 pae"
]

for feature in features: 
    cols = [f"Rep {rep} {feature}" for rep in reps]
    df[f"{feature}_avg"] = df[cols].mean(axis=1)

X = df[[f"{feature}_avg" for feature in features]]
y = df["Known Outcome"]

X.head()

,ptm_avg,iptm_avg,FLS2-flg22 iptm_avg,FLS2-flg22 pae_avg,BAK1-flg22 iptm_avg,BAK1-flg22 pae_avg
0,0.890000,0.913333,0.906667,1.686667,0.696667,2.096667
1,0.883333,0.910000,0.896667,2.023333,0.673333,2.596667
2,0.883333,0.906667,0.860000,2.336667,0.633333,2.663333
3,0.883333,0.906667,0.880000,2.140000,0.650000,2.636667
4,0.880000,0.893333,0.830000,2.996667,0.570000,4.323333


In [3]:
X.describe()

,ptm_avg,iptm_avg,FLS2-flg22 iptm_avg,FLS2-flg22 pae_avg,BAK1-flg22 iptm_avg,BAK1-flg22 pae_avg
count,214.000000,214.000000,214.000000,214.000000,214.000000,214.000000
mean,0.863115,0.816651,0.683053,5.045530,0.403847,8.440763
std,0.031113,0.114895,0.170112,3.572092,0.176854,5.834647
min,0.750000,0.366667,0.303333,1.103333,0.060000,1.843333
25%,0.860000,0.804167,0.536667,2.222500,0.260000,3.958333
50%,0.870000,0.856667,0.728333,3.628333,0.421667,7.213333
75%,0.883333,0.885833,0.835000,6.696667,0.540000,10.759167
max,0.900000,0.920000,0.913333,17.230000,0.706667,27.173333


---
### Initial Feature Ranking

##### ANOVA F-test
"Does this feature have significantly different values between the two classes?"

In [4]:
selector = SelectKBest(score_func=f_classif, k='all')
selector.fit(X,y)

scores = pd.DataFrame({
    "Feature": features,
    "Score": selector.scores_
}).sort_values(by="Score", ascending=False)

print(scores)

           Feature       Score
2  FLS2-flg22 iptm  118.141913
3   FLS2-flg22 pae   69.917917
4  BAK1-flg22 iptm   66.007370
5   BAK1-flg22 pae   16.168699
1             iptm    6.886123
0              ptm    3.012146


##### Logistic Regression Coefficients

In [5]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = LogisticRegression(max_iter=1000)
model.fit(X_scaled, y)

importance = pd.DataFrame({
    "Feature": features,
    "Coefficient": model.coef_[0]
}).sort_values(by="Coefficient", key=abs, ascending=False)

print(importance)

           Feature  Coefficient
4  BAK1-flg22 iptm     1.684086
2  FLS2-flg22 iptm     1.425995
0              ptm    -0.682104
5   BAK1-flg22 pae     0.641964
3   FLS2-flg22 pae    -0.530220
1             iptm    -0.192515


##### Permutation Importance

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

result = permutation_importance(model, X_test, y_test, n_repeats=20, random_state=42)

perm_importance = pd.DataFrame({
    "Feature": features,
    "Importance": result.importances_mean
}).sort_values(by="Importance", ascending=False)

print(perm_importance)

           Feature  Importance
3   FLS2-flg22 pae    0.302326
5   BAK1-flg22 pae    0.004651
0              ptm    0.000000
1             iptm    0.000000
2  FLS2-flg22 iptm   -0.015116
4  BAK1-flg22 iptm   -0.016279


---
### Initial Model Testing

##### Using all features:

In [7]:
# Cross-validation setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000))
    ]),
    
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(probability=True))
    ]),
    
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=5))
    ]),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=3,
        random_state=42
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )
}

In [8]:
# Define scoring metrics
scoring = {
    "roc_auc": "roc_auc",
    "balanced_accuracy": "balanced_accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

results = []

for name, model in models.items():
    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        return_train_score=False
    )
    
    # Store mean and std for each metric
    result = {
        "Model": name,
        
        "ROC-AUC": scores["test_roc_auc"].mean(),
        "ROC-AUC std": scores["test_roc_auc"].std(),
        
        "Balanced Acc": scores["test_balanced_accuracy"].mean(),
        "Balanced Acc std": scores["test_balanced_accuracy"].std(),
        
        "Precision": scores["test_precision"].mean(),
        "Precision std": scores["test_precision"].std(),
        
        "Recall": scores["test_recall"].mean(),
        "Recall std": scores["test_recall"].std(),
        
        "F1": scores["test_f1"].mean(),
        "F1 std": scores["test_f1"].std()
    }
    
    results.append(result)

# Convert to DataFrame
results_df = pd.DataFrame(results)

# Sort by ROC-AUC 
results_df = results_df.sort_values(by="ROC-AUC", ascending=False)

# Print nicely
results_df.head(6)

,Model,ROC-AUC,ROC-AUC std,Balanced Acc,Balanced Acc std,Precision,Precision std,Recall,Recall std,F1,F1 std
1,SVM,0.913285,0.061892,0.833371,0.089115,0.797542,0.069166,0.748352,0.162042,0.768437,0.120862
0,Logistic Regression,0.913038,0.056867,0.849552,0.081532,0.788425,0.076324,0.794505,0.150015,0.786905,0.111731
2,Random Forest,0.909110,0.046770,0.839776,0.072434,0.787937,0.075223,0.774725,0.136635,0.775950,0.098169
5,Gradient Boosting,0.895697,0.039268,0.806789,0.063815,0.752747,0.056649,0.729670,0.159488,0.726836,0.081167
3,KNN,0.887176,0.074977,0.829488,0.081620,0.789177,0.095678,0.747253,0.139045,0.765100,0.111985
4,Decision Tree,0.853013,0.072669,0.810975,0.020944,0.791981,0.087783,0.717582,0.083949,0.743882,0.020244


##### Using only FLS2-flg22 ipTM:

In [9]:
# Cross-validation setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000))
    ]),
    
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(probability=True))
    ]),
    
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=5))
    ]),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=3,
        random_state=42
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )
}

In [10]:
X_reduced = pd.DataFrame(X["FLS2-flg22 iptm_avg"])

# Define scoring metrics
scoring = {
    "roc_auc": "roc_auc",
    "balanced_accuracy": "balanced_accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

results = []

for name, model in models.items():
    scores = cross_validate(
        model,
        X_reduced, #features reduced to only ipTM
        y,
        cv=cv,
        scoring=scoring,
        return_train_score=False
    )
    
    # Store mean and std for each metric
    result = {
        "Model": name,
        
        "ROC-AUC": scores["test_roc_auc"].mean(),
        "ROC-AUC std": scores["test_roc_auc"].std(),
        
        "Balanced Acc": scores["test_balanced_accuracy"].mean(),
        "Balanced Acc std": scores["test_balanced_accuracy"].std(),
        
        "Precision": scores["test_precision"].mean(),
        "Precision std": scores["test_precision"].std(),
        
        "Recall": scores["test_recall"].mean(),
        "Recall std": scores["test_recall"].std(),
        
        "F1": scores["test_f1"].mean(),
        "F1 std": scores["test_f1"].std()
    }
    
    results.append(result)

# Convert to DataFrame
results_df = pd.DataFrame(results)

# Sort by ROC-AUC 
results_df = results_df.sort_values(by="ROC-AUC", ascending=False)

# Print nicely
results_df.head(6)

,Model,ROC-AUC,ROC-AUC std,Balanced Acc,Balanced Acc std,Precision,Precision std,Recall,Recall std,F1,F1 std
0,Logistic Regression,0.910147,0.048293,0.830719,0.077222,0.767493,0.070391,0.763736,0.135571,0.763557,0.104897
1,SVM,0.892895,0.073475,0.834716,0.082594,0.779654,0.076800,0.764835,0.143001,0.769884,0.110806
3,KNN,0.889530,0.056864,0.836292,0.091300,0.757143,0.051464,0.781319,0.183763,0.761491,0.124059
2,Random Forest,0.881925,0.056971,0.801163,0.051766,0.757959,0.050837,0.704396,0.095825,0.727886,0.066403
5,Gradient Boosting,0.879252,0.052392,0.815998,0.052254,0.764835,0.054413,0.734066,0.091519,0.747578,0.068414
4,Decision Tree,0.871497,0.049927,0.828714,0.076539,0.760952,0.021114,0.765934,0.174223,0.751857,0.104492
